# 08 — Nested cross-validation (all models)
Every other comparison in this project uses fixed, untuned
hyperparameters. Here each model's hyperparameters are tuned *inside*
the evaluation, so the reported score includes the cost (and noise) of
tuning on so few cases:

- **Outer loop:** repeated stratified 5-fold CV (3 repeats, 15 test
  folds) on all 157 rows — the same outer folds for every model, so
  their scores can be compared pairwise.
- **Inner loop:** stratified 3-fold grid search on each outer training
  part only, choosing hyperparameters by PR-AUC. Outer test folds are
  never seen during tuning.

This replaces the earlier one-off nested run for A1, which wasn't
captured in a notebook. The prior-only dummy has nothing to tune, so
its chance level is shown as the base rate.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import warnings
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone

from src.feature_engineering import CANDIDATE_FEATURES, NUMERIC_FEATURES, NOMINAL_FEATURES, TARGET_COLUMN, POS_LABEL
from src.evaluation import (
    nested_cross_validate, corrected_ci, corrected_resampled_ttest, cv_split_sizes
)
from src.robustness import build_study_models, model_family
from src.reporting import save_table

warnings.filterwarnings("ignore")

df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "tracer_study_processed.csv")
X, y = df[CANDIDATE_FEATURES], df[TARGET_COLUMN].astype(int)

OUTER_SPLITS, OUTER_REPEATS, INNER_SPLITS = 5, 3, 3
N_TRAIN, N_TEST = cv_split_sizes(len(y), OUTER_SPLITS)
BASE_RATE = (y == POS_LABEL).mean()

## Hyperparameter grids

Kept deliberately small: with ~8 unemployed graduates per inner fold,
large grids mostly fit noise.

In [2]:
PARAM_GRIDS = {
    "BL2_LogReg_Balanced": {"classifier__C": [0.01, 0.1, 1, 10]},
    "A1_RandomForest_ClassWeighted": {"classifier__max_depth": [None, 3, 6],
                                      "classifier__min_samples_leaf": [1, 5]},
    "A2_XGBoost_ScalePosWeight": {"classifier__max_depth": [2, 4],
                                  "classifier__learning_rate": [0.05, 0.3]},
    "A3_SMOTE_RandomForest": {"classifier__max_depth": [None, 3, 6],
                              "resample__k_neighbors": [3, 5]},
    "A4_ADASYN_RandomForest": {"classifier__max_depth": [None, 3, 6],
                               "resample__n_neighbors": [3, 5]},
    "B1_IsolationForest": {"classifier__detector__max_samples": ["auto", 0.5],
                           "classifier__detector__max_features": [1.0, 0.5]},
    "B2_OneClassSVM": {"classifier__detector__nu": [0.1, 0.3, 0.5],
                       "classifier__detector__gamma": ["scale", 0.01, 0.1]},
    "B3_LocalOutlierFactor": {"classifier__detector__n_neighbors": [5, 10, 20, 35]},
}

pd.DataFrame({"model": list(PARAM_GRIDS),
              "grid": [str(g) for g in PARAM_GRIDS.values()]})

,model,grid
0,BL2_LogReg_Balanced,"{'classifier__C': [0.01, 0.1, 1, 10]}"
1,A1_RandomForest_ClassWeighted,"{'classifier__max_depth': [None, 3, 6], 'class..."
2,A2_XGBoost_ScalePosWeight,"{'classifier__max_depth': [2, 4], 'classifier_..."
3,A3_SMOTE_RandomForest,"{'classifier__max_depth': [None, 3, 6], 'resam..."
4,A4_ADASYN_RandomForest,"{'classifier__max_depth': [None, 3, 6], 'resam..."
5,B1_IsolationForest,"{'classifier__detector__max_samples': ['auto',..."
6,B2_OneClassSVM,"{'classifier__detector__nu': [0.1, 0.3, 0.5], ..."
7,B3_LocalOutlierFactor,"{'classifier__detector__n_neighbors': [5, 10, ..."


## Run

In [3]:
templates = build_study_models(NUMERIC_FEATURES, NOMINAL_FEATURES, y_train=y)

nested_scores, chosen = {}, []
for name, grid in PARAM_GRIDS.items():
    template = templates[name]
    scores, params = nested_cross_validate(
        lambda t=template: clone(t),
        grid, X, y, outer_splits=OUTER_SPLITS, outer_repeats=OUTER_REPEATS,
        inner_splits=INNER_SPLITS, return_best_params=True
    )
    nested_scores[name] = scores
    chosen += [{"model": name, "outer_fold": i, **p} for i, p in enumerate(params)]
    print(f"{name}: PR-AUC {scores.mean():.3f}")

BL2_LogReg_Balanced: PR-AUC 0.331


A1_RandomForest_ClassWeighted: PR-AUC 0.321


A2_XGBoost_ScalePosWeight: PR-AUC 0.292


A3_SMOTE_RandomForest: PR-AUC 0.318


A4_ADASYN_RandomForest: PR-AUC 0.319


B1_IsolationForest: PR-AUC 0.270


B2_OneClassSVM: PR-AUC 0.247


B3_LocalOutlierFactor: PR-AUC 0.247


## Results

In [4]:
rows = []
for name, scores in nested_scores.items():
    low, high = corrected_ci(scores, N_TRAIN, N_TEST)
    rows.append({"model": name, "family": model_family(name), "pr_auc": scores.mean(),
                 "pr_auc_std": scores.std(), "pr_auc_corrected_ci_low": low,
                 "pr_auc_corrected_ci_high": high, "base_rate": BASE_RATE,
                 "n_outer_folds": len(scores)})
nested_summary = pd.DataFrame(rows).sort_values("pr_auc", ascending=False)

untuned = pd.read_csv(PROJECT_ROOT / "reports" / "tables" / "robustness_summary.csv")
untuned = untuned[untuned["scenario"].str.startswith("reference")].set_index("model")["pr_auc"]
nested_summary["untuned_full_data_pr_auc"] = nested_summary["model"].map(untuned)

display(nested_summary.round(3))
save_table(nested_summary, "nested_cv_summary", index=False)

chosen_params = pd.DataFrame(chosen)
save_table(chosen_params, "nested_cv_best_params", index=False)
print("Hyperparameters chosen per outer fold (how often each value won):")
for name in PARAM_GRIDS:
    sub = chosen_params[chosen_params["model"] == name].drop(columns=["model", "outer_fold"]).dropna(axis=1)
    print(f"  {name}: " + "; ".join(f"{c.split('__')[-1]} {sub[c].astype(str).value_counts().to_dict()}" for c in sub))

,model,family,pr_auc,pr_auc_std,pr_auc_corrected_ci_low,pr_auc_corrected_ci_high,base_rate,n_outer_folds,untuned_full_data_pr_auc
0,BL2_LogReg_Balanced,Baseline,0.331,0.104,0.201,0.460,0.197,15,0.361
1,A1_RandomForest_ClassWeighted,A,0.321,0.091,0.208,0.435,0.197,15,0.302
4,A4_ADASYN_RandomForest,A,0.319,0.084,0.215,0.424,0.197,15,0.301
3,A3_SMOTE_RandomForest,A,0.318,0.106,0.186,0.451,0.197,15,0.309
2,A2_XGBoost_ScalePosWeight,A,0.292,0.085,0.186,0.398,0.197,15,0.266
5,B1_IsolationForest,B,0.270,0.073,0.179,0.362,0.197,15,0.275
7,B3_LocalOutlierFactor,B,0.247,0.068,0.162,0.332,0.197,15,0.253
6,B2_OneClassSVM,B,0.247,0.078,0.150,0.344,0.197,15,0.228


Hyperparameters chosen per outer fold (how often each value won):
  BL2_LogReg_Balanced: C {'0.01': 8, '10.0': 6, '0.1': 1}
  A1_RandomForest_ClassWeighted: min_samples_leaf {'5.0': 10, '1.0': 5}
  A2_XGBoost_ScalePosWeight: max_depth {'2.0': 9, '4.0': 6}; learning_rate {'0.05': 12, '0.3': 3}
  A3_SMOTE_RandomForest: k_neighbors {'3.0': 10, '5.0': 5}
  A4_ADASYN_RandomForest: n_neighbors {'3.0': 9, '5.0': 6}
  B1_IsolationForest: max_features {'1.0': 9, '0.5': 6}; max_samples {'0.5': 13, 'auto': 2}
  B2_OneClassSVM: gamma {'0.01': 6, 'scale': 6, '0.1': 3}; nu {'0.5': 15}
  B3_LocalOutlierFactor: n_neighbors {'35.0': 12, '10.0': 1, '20.0': 1, '5.0': 1}


## Pairwise tests on the outer folds

Corrected resampled t-test on the 15 outer-fold scores. Two families of
comparisons, each Bonferroni-corrected separately: every study model vs.
logistic regression (7 tests), and Family A vs. Family B (12 tests).

In [5]:
def tests(pairs):
    out = []
    for a, b in pairs:
        t_stat, p = corrected_resampled_ttest(nested_scores[a], nested_scores[b], N_TRAIN, N_TEST)
        out.append({"model_a": a, "model_b": b,
                    "mean_diff": nested_scores[a].mean() - nested_scores[b].mean(),
                    "statistic": t_stat, "p_value": p})
    out = pd.DataFrame(out)
    out["significant"] = out["p_value"] < 0.05 / len(out)
    return out

study = [m for m in nested_scores if model_family(m) in ("A", "B")]
vs_logreg = tests([(m, "BL2_LogReg_Balanced") for m in study]).assign(comparison="vs logistic regression")
cross = tests([(a, b) for a in study if model_family(a) == "A"
                      for b in study if model_family(b) == "B"]).assign(comparison="Family A vs. B")
nested_tests = pd.concat([vs_logreg, cross])
display(nested_tests.round(4))
save_table(nested_tests, "nested_cv_tests", index=False);

,model_a,model_b,mean_diff,statistic,p_value,significant,comparison
0,A1_RandomForest_ClassWeighted,BL2_LogReg_Balanced,-0.0094,-0.1606,0.8747,False,vs logistic regression
1,A2_XGBoost_ScalePosWeight,BL2_LogReg_Balanced,-0.0393,-0.7329,0.4757,False,vs logistic regression
2,A3_SMOTE_RandomForest,BL2_LogReg_Balanced,-0.0124,-0.1757,0.8630,False,vs logistic regression
3,A4_ADASYN_RandomForest,BL2_LogReg_Balanced,-0.0115,-0.1595,0.8756,False,vs logistic regression
4,B1_IsolationForest,BL2_LogReg_Balanced,-0.0607,-1.5234,0.1499,False,vs logistic regression
5,B2_OneClassSVM,BL2_LogReg_Balanced,-0.0840,-1.9121,0.0765,False,vs logistic regression
6,B3_LocalOutlierFactor,BL2_LogReg_Balanced,-0.0839,-2.0488,0.0597,False,vs logistic regression
0,A1_RandomForest_ClassWeighted,B1_IsolationForest,0.0513,0.8700,0.3990,False,Family A vs. B
1,A1_RandomForest_ClassWeighted,B2_OneClassSVM,0.0745,1.2736,0.2236,False,Family A vs. B
2,A1_RandomForest_ClassWeighted,B3_LocalOutlierFactor,0.0745,1.2811,0.2210,False,Family A vs. B
